# [U1 - Lab] Ingesta y Montaje de Archivos en Google Drive

Ejecuta primero la celda 1 (módulo) y luego la celda 2 (ejecución).

In [ ]:
"""
[U1 - Lab] Ingesta y Montaje de Archivos en Google Drive

Módulo estructurado que monta Google Drive (en Google Colab), explora un
directorio configurado mediante constantes y procesa archivos de datos.
"""

import json
import os
from datetime import datetime
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    display = print

# ---------------------------------------------------------------------------
# Constantes de configuración
# ---------------------------------------------------------------------------
PUNTO_MONTAJE = '/content/drive'
DIRECTORIO = '/content/drive/My Drive/Lenguaje II'
EXTENSIONES_PERMITIDAS = ('.json', '.csv')
ARCHIVO_INVENTARIO = 'inventario.json'


def montar_drive():
    """Monta Google Drive si se ejecuta dentro de Google Colab."""
    try:
        from google.colab import drive
        drive.mount(PUNTO_MONTAJE)
        return True
    except ImportError:
        print('Aviso: no se está ejecutando en Google Colab; se omite el montaje.')
        return False


def obtener_metadatos(ruta):
    """Devuelve tamaño en KB y fecha de última modificación de un archivo."""
    tamano_kb = round(os.path.getsize(ruta) / 1024, 2)
    fecha = datetime.fromtimestamp(os.path.getmtime(ruta))
    return tamano_kb, fecha.strftime('%Y-%m-%d %H:%M:%S')


def leer_inventario(ruta):
    """
    Lee inventario.json con json.load() y valida que sea una lista de
    registros (cada registro debe ser un diccionario).
    Retorna (datos, es_valido, mensaje).
    """
    try:
        with open(ruta, 'r', encoding='utf-8') as f:
            datos = json.load(f)
    except json.JSONDecodeError as e:
        return None, False, f'JSON mal formado: {e}'
    except OSError as e:
        return None, False, f'No se pudo leer el archivo: {e}'

    if not isinstance(datos, list):
        return datos, False, 'El contenido no es una lista de registros.'
    if not all(isinstance(r, dict) for r in datos):
        return datos, False, 'La lista contiene elementos que no son registros (dict).'
    return datos, True, f'Inventario válido con {len(datos)} registros.'


def list_files_in_directory(directorio=DIRECTORIO, extensiones=EXTENSIONES_PERMITIDAS):
    """
    Explora un directorio, filtra archivos por extensión, calcula metadatos
    y parsea inventario.json si existe.

    Retorna un diccionario estructurado, listo para ser consumido por una API.
    """
    resultado = {
        'directorio': directorio,
        'total_archivos': 0,
        'archivos': [],
        'inventario': {
            'encontrado': False,
            'valido': False,
            'mensaje': 'No se encontró inventario.json en la carpeta.',
            'registros': None,
        },
        'error': None,
    }

    try:
        nombres = sorted(os.listdir(directorio))
    except FileNotFoundError:
        print('Directorio no encontrado.')
        resultado['error'] = 'Directorio no encontrado.'
        return resultado
    except PermissionError:
        print('Sin permisos para acceder al directorio.')
        resultado['error'] = 'Sin permisos para acceder al directorio.'
        return resultado

    print(f'Archivos en {directorio}:')
    for nombre in nombres:
        ruta = os.path.join(directorio, nombre)
        if not os.path.isfile(ruta):
            continue
        extension = os.path.splitext(nombre)[1].lower()
        if extension not in extensiones:
            continue

        tamano_kb, modificado = obtener_metadatos(ruta)
        print(f'  └── {nombre} | {tamano_kb} KB | modificado: {modificado}')
        resultado['archivos'].append({
            'nombre': nombre,
            'extension': extension,
            'tamano_kb': tamano_kb,
            'ultima_modificacion': modificado,
        })

    resultado['total_archivos'] = len(resultado['archivos'])

    ruta_inv = os.path.join(directorio, ARCHIVO_INVENTARIO)
    if os.path.isfile(ruta_inv):
        datos, valido, mensaje = leer_inventario(ruta_inv)
        resultado['inventario'] = {
            'encontrado': True,
            'valido': valido,
            'mensaje': mensaje,
            'registros': datos if valido else None,
        }
        print(f'Inventario: {mensaje}')

    return resultado


In [ ]:
montar_drive()
datos = list_files_in_directory(DIRECTORIO)
print('\nResultado estructurado:')
print(json.dumps(datos, indent=2, ensure_ascii=False))

print("\n--- Visualización en Tabla ---")
if datos['archivos']:
    df_archivos = pd.DataFrame(datos['archivos'])
    display(df_archivos)
else:
    print("No se encontraron archivos con las extensiones permitidas.")
